# flag column analysis

## Load data

In [ ]:
import re
import pandas as pd

LOG_PATH = "../data/original_dataset/HPC.log"
COLUMNS = ["event_id", "node", "component", "state", "time", "flag", "message"]

LINE_RE = re.compile(r"^(\d+) (.*?) (\S+) (\S+) (\d+(?:\.\d+)?) (-?\d)(?: (.*))?$")


def load_hpc_log(path=LOG_PATH):
    with open(path, "rb") as f:
        lines = [l for l in f.read().decode("ascii").split("\r\n") if l]

    rows, unparsed = [], []
    for line in lines:
        m = LINE_RE.match(line)
        if m:
            rows.append(m.groups())
        else:
            unparsed.append(line)
    
    df = pd.DataFrame(rows, columns=COLUMNS)
    df["message"] = df["message"].fillna("")
    df["event_id"] = df["event_id"].astype("int64")
    df["flag"] = df["flag"].astype("int8")
    t = pd.to_numeric(df["time"], errors="coerce")
    df["bad_time"] = ~df["time"].str.fullmatch(r"\d{10}")
    df["ts"] = pd.to_datetime(t.where(~df["bad_time"]), unit="s")
    df["time"] = t
    df["row"] = range(len(df))
    return df, unparsed

In [2]:
df, unparsed = load_hpc_log()
print(f"{len(df):,} rows, {len(unparsed)} unparsed, {df.bad_time.sum()} bad timestamps")
df.head()

433,490 rows, 0 unparsed, 9 bad timestamps


,event_id,node,component,state,time,flag,message,bad_time,ts,row
0,460903,resourcemgmtdaeomon node-25,server,subsys,1.145552e+09,1,failed to configure resourcemgmt subsystem err...,False,2006-04-20 16:56:56,0
1,460919,resourcemgmtdaeomon node-25,server,subsys,1.145552e+09,1,failed to configure resourcemgmt subsystem err...,False,2006-04-20 16:57:01,1
2,460932,resourcemgmtdaeomon node-25,server,subsys,1.145552e+09,1,failed to configure resourcemgmt subsystem err...,False,2006-04-20 16:57:06,2
3,460949,resourcemgmtdaeomon node-25,server,subsys,1.145552e+09,1,failed to configure resourcemgmt subsystem err...,False,2006-04-20 16:57:12,3
4,460971,resourcemgmtdaeomon node-25,server,subsys,1.145552e+09,1,failed to configure resourcemgmt subsystem err...,False,2006-04-20 16:57:17,4


## flag values

In [3]:
df.flag.value_counts()

flag
 1    418546
 0     11779
-1      3165
Name: count, dtype: int64

## flag by component and state

In [4]:
t = df.groupby(["component", "state", "flag"]).size().unstack("flag", fill_value=0)
t[(t[0] > 0) | (t[-1] > 0)].sort_values(by=[-1, 0, 1], ascending=False)

flag                         -1      0       1
component     state                           
partition     status       2083     36    5423
node          status       1045  11516   45425
switch_module error          34    134  104423
              control         3      0     528
node          temperature     0     73   54669
domain        status          0     20    1438

## flag by message
All messages (numbers replaced by placeholders) that ever appear with flag 0 or -1.

In [5]:
msg = df.message.str.replace(r"0x[0-9a-fA-F]+", "<HEX>", regex=True).str.replace(r"\d+", "<N>", regex=True)
m = df.groupby(["component", "state", msg, "flag"]).size().unstack("flag", fill_value=0)
print(len(m), "message patterns in total,", ((m[0] > 0) | (m[-1] > 0)).sum(), "with flag 0 or -1")
m[(m[0] > 0) | (m[-1] > 0)].sort_values(by=[-1, 0, 1], ascending=False)

403 message patterns in total, 15 with flag 0 or -1


flag                                                                            -1  \
component     state       message                                                    
partition     status      running                                             2015   
node          status      configured out                                      1011   
partition     status      blocked                                               68   
node          status      active                                                34   
switch_module error       link errors remain current                            34   
              control     power/control ok                                       3   
node          status      running                                                0   
                          not responding                                         0   
switch_module error       Linkerror event interval expired                       0   
node          temperature ambient=<N>                                            0   
domain        status      not responding                                         0   
                          inconsistent nodesets node-<N> <HEX> <ok> node-...     0   
node          temperature                                                        0   
partition     status      down                                                   0   
                          starting                                               0   

flag                                                                             0  \
component     state       message                                                    
partition     status      running                                               16   
node          status      configured out                                      3195   
partition     status      blocked                                               17   
node          status      active                                               122   
switch_module error       link errors remain current                             2   
              control     power/control ok                                       0   
node          status      running                                             7217   
                          not responding                                       982   
switch_module error       Linkerror event interval expired                     132   
node          temperature ambient=<N>                                           70   
domain        status      not responding                                        16   
                          inconsistent nodesets node-<N> <HEX> <ok> node-...     4   
node          temperature                                                        3   
partition     status      down                                                   2   
                          starting                                               1   

flag                                                                              1  
component     state       message                                                    
partition     status      running                                              2910  
node          status      configured out                                      15633  
partition     status      blocked                                              2201  
node          status      active                                               1107  
switch_module error       link errors remain current                          13609  
              control     power/control ok                                       78  
node          status      running                                             17906  
                          not responding                                      10724  
switch_module error       Linkerror event interval expired                    84762  
node          temperature ambient=<N>                                         54669  
domain        status      not responding  

## Identical copies of the same row, by flag
`copies` = number of rows with the same node, component, state, time and message.

In [6]:
key = ["node", "component", "state", "time", "message"]
df["copies"] = df.groupby(key).flag.transform("size")
df.groupby("flag").copies.describe()

,count,mean,std,min,25%,50%,75%,max
flag,,,,,,,,
-1,3165.0,12.414218,11.208319,1.0,4.0,4.0,24.0,34.0
0,11779.0,3.505646,1.554860,1.0,2.0,3.0,5.0,25.0
1,418546.0,1.064399,0.314302,1.0,1.0,1.0,1.0,12.0


In [7]:
df.groupby("flag").copies.apply(lambda x: (x == 1).mean()).rename("share of rows with no duplicate")

flag
-1    0.007267
 0    0.086849
 1    0.944606
Name: share of rows with no duplicate, dtype: float64

## Which flags appear together among identical copies

In [8]:
counts = pd.get_dummies(df.flag).rename(columns={1: "flag_1", 0: "flag_0", -1: "flag_neg1"})
groups = pd.concat([df[key], counts], axis=1).groupby(key).sum()
groups[["flag_1", "flag_0", "flag_neg1"]].gt(0).value_counts()

flag_1  flag_0  flag_neg1
True    False   False        405251
        True    False          3287
False   True    False          2390
        False   True            554
        True    True              1
Name: count, dtype: int64

## Groups that mix flag 0 and flag 1: how many flag 1 copies do they have

In [9]:
groups[(groups.flag_0 > 0) & (groups.flag_1 > 0)].flag_1.value_counts()

flag_1
1    3287
Name: count, dtype: int64

## Order of the flags among identical copies, in file order
Each group's flags as they appear in the file, with repeated values collapsed.

In [10]:
copied = df[df.copies > 1].sort_values("row")
flag_order = copied.groupby(key).flag.agg(lambda f: tuple(f[f != f.shift()]))
flag_order.value_counts()

flag
(1,)       9890
(0, 1)     3287
(0,)       1367
(-1,)       531
(0, -1)       1
Name: count, dtype: int64

## Flag of the last line of each distinct event

In [11]:
df.sort_values("row").groupby(key).flag.last().value_counts()

flag
 1    408538
 0      2390
-1       555
Name: count, dtype: int64

## Distinct events and rows that carry flag 0 or -1

In [12]:
has_0_or_neg1 = (groups.flag_0 > 0) | (groups.flag_neg1 > 0)
print(f"{has_0_or_neg1.sum():,} of {len(groups):,} distinct events carry flag 0 or -1")
print(f"{(df.flag != 1).sum():,} of {len(df):,} rows carry flag 0 or -1")

6,232 of 411,483 distinct events carry flag 0 or -1
14,944 of 433,490 rows carry flag 0 or -1


## Example: one node around the moment with the most flag 0 rows

In [13]:
top_time = df[df.flag == 0].groupby("time").size().idxmax()
example_node = df[(df.time == top_time) & (df.flag == 0)].node.iloc[0]
window = df[(df.node == example_node) & df.time.between(top_time - 86400, top_time + 86400)]
window.sort_values("time")[["event_id", "node", "component", "state", "ts", "flag", "message"]]

,event_id,node,component,state,ts,flag,message
121349,442412,node-103,node,status,2004-10-21 13:56:00,1,not responding
121440,442503,node-103,node,status,2004-10-21 13:56:30,1,configured out
19965,442979,node-103,action,start,2004-10-21 20:09:19,1,boot (command 3197)
20029,443072,node-103,action,start,2004-10-21 20:12:30,1,wait (command 3197)
121640,448745,node-103,node,status,2004-10-21 21:41:02,0,running
122091,448745,node-103,node,status,2004-10-21 21:41:02,0,running
122447,448745,node-103,node,status,2004-10-21 21:41:02,0,running
122855,448745,node-103,node,status,2004-10-21 21:41:02,0,running


## Example: one node around the moment with the most flag -1 rows

In [14]:
top_time = df[df.flag == -1].groupby("time").size().idxmax()
example_node = df[(df.time == top_time) & (df.flag == -1)].node.iloc[0]
window = df[(df.node == example_node) & df.time.between(top_time - 86400, top_time + 86400)]
window.sort_values("time")[["event_id", "node", "component", "state", "ts", "flag", "message"]]

,event_id,node,component,state,ts,flag,message
113496,371667,node-224,node,status,2004-05-30 12:48:31,1,not responding
113125,372041,node-224,node,status,2004-05-30 12:49:00,-1,configured out
113908,372041,node-224,node,status,2004-05-30 12:49:00,-1,configured out
113866,372041,node-224,node,status,2004-05-30 12:49:00,-1,configured out
114101,372041,node-224,node,status,2004-05-30 12:49:00,-1,configured out
114497,372041,node-224,node,status,2004-05-30 12:49:00,-1,configured out
115085,372041,node-224,node,status,2004-05-30 12:49:00,-1,configured out
114794,372041,node-224,node,status,2004-05-30 12:49:00,-1,configured out
116556,372041,node-224,node,status,2004-05-30 12:49:00,-1,configured out
116641,372041,node-224,node,status,2004-05-30 12:49:00,-1,configured out


## flag over time

In [15]:
pd.crosstab(df.ts.dt.to_period("Q"), df.flag)

flag,-1,0,1
ts,,,
2003Q3,1932,0,0
2003Q4,0,0,3930
2004Q1,26,2484,135292
2004Q2,832,2981,52543
2004Q3,57,1362,23606
2004Q4,0,918,20523
2005Q1,0,154,33458
2005Q2,0,1416,29106
2005Q3,306,1028,37132


## Conclusions
- `flag` takes the values 1 (418,546 rows), 0 (11,779) and -1 (3,165). It is undocumented, so these are observations, not a confirmed meaning.
- Rows with flag 0 or -1 are almost always repeated copies of the same row (same node, time, message and `event_id`). Only 8.7% of flag 0 rows and 0.7% of flag -1 rows have no duplicate, against 94.5% of flag 1 rows. The median number of copies is 3 for flag 0, 4 for flag -1 and 1 for flag 1.
- When identical copies mix flag 0 and flag 1 (3,287 groups), exactly one copy has flag 1 and the rest have flag 0.
- Flag -1 never shares a group with flag 1.
- `flag` does not track severity. Flags 0 and -1 appear on 15 of 403 message patterns, almost all of them routine state reports (`running`, `configured out`, `not responding`, `blocked`, `active`), and each of those messages also appears with flag 1.
- After collapsing identical copies, 6,232 of 411,483 distinct events carry flag 0 or -1.
- Decision: do not use `flag` as an error label. Drop identical copies before any analysis, which removes most of the flag 0 and -1 rows.